# EvoFuse v4: Grammatical Evolution for Multimodal Fusion Architecture Search

**Multi-Dataset Cancer Detection Benchmark — Publication-Quality Pipeline**

*Datasets: WBCD (pseudo-modality split reference), TCGA-KIPAN (real multi-omics: mRNA + DNA methylation) and MFEAT (real multi-view: profile correlations + Fourier coefficients).*

*EvoFuse is the successor to GE-MFAS v3, rebuilt after a root-cause analysis of why
GE-MFAS underperformed its baselines. Every change below maps to a diagnosed cause (C1–C9):*

| # | GE-MFAS v3 problem | EvoFuse v4 fix |
|---|--------------------|----------------|
| C1 | `StandardScaler`/variance filtering fit on the **full dataset** before CV → information leak | `FoldPreprocessor` fits impute→filter→scale **inside each outer training fold**, shared by all methods |
| C2 | Frozen 512-dim Conv1d "pseudo-sequence" backbones (>100k params on 30 features, n≈500) → wrong inductive bias, capacity mismatch | Compact end-to-end MLP encoders + fusion ops (3k–40k params), trained jointly |
| C3 | Fitness = accuracy on **one ~68-sample split** (1 sample = 1.5%) → selection driven by noise; GE ≈ random search | Fitness = mean **balanced accuracy over 3 fixed stratified splits**, deterministic seeds → paired, low-variance comparisons |
| C4 | "Best-of-3-runs" + best-epoch-by-val selection → winner's curse on the tiny val set | Final predictor = **soft-voting ensemble of top-k distinct architectures** × multiple seeds |
| C5 | Skip-connection code never affected the output (list-index bug) → 1 of 3 grammar genes was a silent no-op | Dead code removed; **self-test proves every gene changes the phenotype** |
| C6 | Plain accuracy for selection → collapses to majority rate on imbalanced data (Cervical 14:1) | Balanced accuracy everywhere + class-weighted loss |
| C7 | `except: return 0.0` swallowed all evaluation errors | Failures logged and counted, surfaced in results |
| C8 | ~3000 evals/fold with massive duplicate re-training | Phenotype-level fitness **cache**, early stopping, and an **equal-budget random-search control** |
| C9 | Global seed only; results irreproducible run-to-run | Per-(phenotype, split) deterministic seeding; config + environment stamped into results |


## 1. Environment Setup

In [ ]:
# -- Install dependencies (run once) --
# !pip install torch numpy pandas scikit-learn matplotlib seaborn scipy
# !pip install xgboost lightgbm
# !pip install mvlearn   # bundles the UCI Multiple Features multi-view benchmark offline

# =============================================================================
# EvoFuse: Grammatical Evolution for Multimodal Fusion Architecture Search
# (formerly GE-MFAS) — v4, rewritten for robustness, reproducibility and
# competitive performance on small-sample multimodal biomedical data.
# =============================================================================
# Key changes vs GE-MFAS v3 (each maps to a diagnosed failure cause):
#   C1. Leak-free evaluation ......... imputation/scaling/variance filtering are
#       fit INSIDE each outer training fold (v3 fit them on the full dataset).
#   C2. Right-sized search space ..... compact end-to-end MLP encoders + fusion
#       ops replace frozen 512-dim conv "pseudo-sequence" backbones whose
#       capacity (>100k params) swamped n≈500 tabular datasets.
#   C3. Low-variance fitness ......... mean balanced accuracy over 3 fixed
#       stratified inner splits (v3: one ~68-sample split → 1 sample = 1.5% acc,
#       so selection was driven by noise and GE could not beat random search).
#   C4. Ensemble, not best-of-N ...... final predictor = soft-voting ensemble of
#       the top-k distinct evolved architectures (v3's best-of-3-runs selection
#       by val fitness was a winner's-curse amplifier).
#   C5. Dead code removed ............ v3 skip connections never reached the
#       output (list-index bug) and one grammar gene in three was a no-op;
#       every EvoFuse gene verifiably changes the network (see unit test).
#   C6. Imbalance-aware .............. balanced accuracy drives selection and
#       early stopping; class-weighted loss (v3 selected on plain accuracy,
#       which collapses to the majority rate on Cervical's 14:1 imbalance).
#   C7. No silent failures ........... evaluation exceptions are logged and
#       counted (v3 returned fitness 0.0 silently).
#   C8. Compute discipline ........... phenotype-level caching (GE revisits
#       duplicates constantly), early stopping everywhere, and a random-search
#       control given exactly the same unique-evaluation budget.
#   C9. Reproducibility .............. deterministic seeds per (phenotype,
#       split), config + environment captured with every result file.
# =============================================================================

## 2. Configuration & Reproducibility (C9)

In [ ]:
import copy, hashlib, json, math, os, random, time, warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

try:
    import xgboost as xgb; HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    import lightgbm as lgb; HAS_LGB = True
except ImportError:
    HAS_LGB = False
try:
    from scipy import stats as sps; HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

warnings.filterwarnings("ignore")
torch.set_num_threads(max(1, os.cpu_count() or 1))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CFG = {
    # --- Grammatical Evolution ---
    "population_size": 24,
    "max_generations": 15,
    "elite_size": 4,
    "tournament_size": 3,
    "crossover_rate": 0.9,
    "codon_mutation_rate": 0.08,
    "genotype_length": 32,
    "codon_size": 256,
    "ge_patience": 5,            # generations without improvement -> stop
    # --- Fitness evaluation (proxy) ---
    "inner_splits": 3,           # C3: fitness = mean over 3 stratified splits
    "inner_val_frac": 0.25,
    "proxy_max_epochs": 80,
    "proxy_patience": 10,
    # --- Training ---
    "batch_size": 64,
    "lr": 2e-3,
    "weight_decay": 1e-4,
    "label_smoothing": 0.0,
    # --- Final model (C4) ---
    "ensemble_k": 3,             # top-k distinct architectures
    "ensemble_seeds": 2,         # seeds per architecture
    "final_max_epochs": 200,
    "final_patience": 20,
    "final_val_frac": 0.15,
    # --- Outer protocol ---
    "n_outer_folds": 5,
    "n_repetitions": 1,
    "base_seed": 42,
    "device": DEVICE,
}


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 3. Data: Raw Loaders + Leak-Free Per-Fold Preprocessing (C1)

Loaders return **raw, unscaled** arrays (NaNs kept where present). All
imputation, variance filtering and standardisation are fit per outer
training fold by `FoldPreprocessor` and applied unchanged to test data.
TCGA loaders read your local `Datasets/` directory; a clearly-labelled
synthetic generator exists for pipeline testing only.

In [ ]:
def load_wbcd_raw():
    """Wisconsin Breast Cancer — shape vs texture modality split (RAW values).

    Scaling is deliberately NOT applied here: it is fit per outer fold by
    FoldPreprocessor (v3 fit StandardScaler on the full dataset -> leakage).
    """
    data = load_breast_cancer()
    X, y = data.data.astype(np.float64), data.target.astype(np.int64)
    names = list(data.feature_names)
    shape_base, texture_base = [0, 2, 3, 5, 6, 7], [1, 4, 8, 9]
    shape_idx = [o + b for o in (0, 10, 20) for b in shape_base]
    texture_idx = [o + b for o in (0, 10, 20) for b in texture_base]
    info = {"name": "WBCD", "n": len(y), "classes": 2,
            "mod_x_type": "Shape/Morphology", "mod_y_type": "Surface/Texture"}
    return (X[:, shape_idx], X[:, texture_idx], y,
            [names[i] for i in shape_idx], [names[i] for i in texture_idx], info)


def load_cervical_raw(path="Datasets/cervical/risk_factors_cervical_cancer.csv"):
    """UCI Cervical Cancer — demographic vs STD split (RAW values, NaNs kept).

    Missing values are KEPT here and imputed per fold (median fit on train).
    """
    p = Path(path)
    if not p.exists():
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "00383/risk_factors_cervical_cancer.csv")
        df = pd.read_csv(url, na_values="?")
    else:
        df = pd.read_csv(p, na_values="?")
    df = df.dropna(axis=1, thresh=len(df) * 0.5)          # structural, label-free
    target_cols = [c for c in ["Biopsy", "Hinselmann", "Schiller", "Citology"]
                   if c in df.columns]
    y = df["Biopsy"].values.astype(np.int64)
    feats = df.drop(columns=target_cols, errors="ignore")
    demo_kw = ["age", "number of sexual partners", "first sexual intercourse",
               "num of pregnancies", "smokes", "hormonal contraceptives", "iud"]
    std_kw = ["stds", "condylomatosis", "syphilis", "pelvic inflammatory",
              "genital herpes", "molluscum", "hiv", "hepatitis", "hpv", "dx"]
    mod_x_cols, mod_y_cols = [], []
    for c in feats.columns:
        cl = c.lower()
        if any(k in cl for k in std_kw):
            mod_y_cols.append(c)
        elif any(k in cl for k in demo_kw):
            mod_x_cols.append(c)
        else:
            (mod_x_cols if len(mod_x_cols) <= len(mod_y_cols) else mod_y_cols).append(c)
    info = {"name": "Cervical", "n": len(y), "classes": 2,
            "mod_x_type": "Demographic/Behavioural", "mod_y_type": "STD history"}
    return (feats[mod_x_cols].values.astype(np.float64),
            feats[mod_y_cols].values.astype(np.float64), y,
            mod_x_cols, mod_y_cols, info)


def load_tcga_raw(cancer_type, data_dir="Datasets", top_k_rna=500, top_k_mod2=200):
    """TCGA multi-omics (RNA-seq + miRNA/clinical). Feature pre-selection by
    variance is applied here only to bound dimensionality; per-fold variance
    filtering and scaling still happen inside FoldPreprocessor."""
    base = Path(data_dir) / cancer_type.lower().replace("-", "_") / "processed"
    if not base.exists():
        base = Path(data_dir) / cancer_type.lower().replace("-", "_")
    rna_f, lab_f = base / "rnaseq.csv", base / "labels.csv"
    if not rna_f.exists() or not lab_f.exists():
        return None  # caller decides whether to substitute synthetic data
    rna = pd.read_csv(rna_f, index_col=0)
    labels_df = pd.read_csv(lab_f, index_col=0)
    mod2, mod2_name = None, None
    for fname, nm in [("mirna.csv", "miRNA"), ("clinical.csv", "Clinical"),
                      ("methylation.csv", "DNAm")]:
        if (base / fname).exists():
            mod2 = pd.read_csv(base / fname, index_col=0); mod2_name = nm; break
    if mod2 is None:
        return None
    if mod2_name == "Clinical":
        mod2 = mod2.select_dtypes(include=[np.number])
        mod2 = mod2.dropna(axis=1, thresh=len(mod2) * 0.5)
    common = rna.index.intersection(mod2.index).intersection(labels_df.index)
    if len(common) < 50:
        return None
    rna, mod2 = rna.loc[common], mod2.loc[common]
    y = labels_df.loc[common].values.ravel().astype(np.int64)
    if rna.shape[1] > top_k_rna:
        rna = rna[rna.var().nlargest(top_k_rna).index]
    if mod2.shape[1] > top_k_mod2:
        mod2 = mod2[mod2.var().nlargest(top_k_mod2).index]
    info = {"name": cancer_type, "n": len(y), "classes": len(np.unique(y)),
            "mod_x_type": "RNA-seq", "mod_y_type": mod2_name}
    return (rna.values.astype(np.float64), mod2.values.astype(np.float64), y,
            list(rna.columns), list(mod2.columns), info)


def make_synthetic_multiomics(name="Synthetic-Omics", n=500, dim_x=500, dim_y=200,
                              seed=7, noise=2.0):
    """Clearly-labelled synthetic multi-omics stand-in (pipeline testing only)."""
    rng = np.random.RandomState(seed)
    y = rng.randint(0, 2, n).astype(np.int64)
    sx = np.outer(y - 0.5, rng.randn(dim_x))
    sy = np.outer(y - 0.5, rng.randn(dim_y))
    mod_x = (noise * rng.randn(n, dim_x) + sx)
    mod_y = (noise * rng.randn(n, dim_y) + sy)
    info = {"name": f"{name} (synthetic)", "n": n, "classes": 2,
            "mod_x_type": "synthetic RNA-like", "mod_y_type": "synthetic miRNA-like"}
    return (mod_x, mod_y, y,
            [f"gx{i}" for i in range(dim_x)], [f"gy{i}" for i in range(dim_y)], info)


class FoldPreprocessor:
    """Impute -> variance-filter -> standardise, fit ONLY on the training fold
    and applied unchanged to validation/test data (fixes v3's global-fit leak).
    Shared by EvoFuse and every baseline so comparisons are like-for-like."""

    def __init__(self):
        self.imp_x = SimpleImputer(strategy="median")
        self.imp_y = SimpleImputer(strategy="median")
        self.sc_x, self.sc_y = StandardScaler(), StandardScaler()
        self.keep_x = self.keep_y = None

    def fit(self, mod_x_tr, mod_y_tr):
        xi = self.imp_x.fit_transform(mod_x_tr)
        yi = self.imp_y.fit_transform(mod_y_tr)
        self.keep_x = xi.var(axis=0) > 1e-12
        self.keep_y = yi.var(axis=0) > 1e-12
        self.sc_x.fit(xi[:, self.keep_x]); self.sc_y.fit(yi[:, self.keep_y])
        return self

    def transform(self, mod_x, mod_y):
        xi = self.imp_x.transform(mod_x)[:, self.keep_x]
        yi = self.imp_y.transform(mod_y)[:, self.keep_y]
        out_x = self.sc_x.transform(xi).astype(np.float32)
        out_y = self.sc_y.transform(yi).astype(np.float32)
        return np.nan_to_num(out_x), np.nan_to_num(out_y)

def load_kipan_raw(path="Datasets/kipan"):
    """TCGA-KIPAN pan-kidney multi-omics — a REAL two-modality dataset.

    mRNA expression (2000 genes) + DNA methylation (2000 CpG probes) for
    n=707 patients; 3 classes: KICH (n=65), KIRC (n=345), KIRP (n=297).
    Matrices are the MOGONET benchmark files (Wang et al., Nat. Commun.
    12:3445, 2021), redistributed at github.com/gabrieletaz/MKL_MO
    (data/KIPAN/seed_0; train/test halves concatenated because this
    pipeline runs its own outer CV). Values arrive [0,1]-scaled from the
    benchmark preprocessing; per-fold standardisation is still applied by
    FoldPreprocessor, fit on each outer training fold only.

    Caveat (worth stating in a paper): the benchmark's 2000-feature
    pre-selection was performed by the MOGONET authors on their fixed
    training split, so a small selection bias relative to fully
    fold-internal selection is inherited — the standard situation for
    every method benchmarked on these widely reused files.
    """
    base = Path(path)
    rna = pd.read_csv(base / "mrna.csv", index_col=0)
    meth = pd.read_csv(base / "methylation.csv", index_col=0)
    y = pd.read_csv(base / "labels.csv", index_col=0).values.ravel().astype(np.int64)
    assert len(rna) == len(meth) == len(y), "modality/label row mismatch"
    info = {"name": "KIPAN", "n": len(y), "classes": len(np.unique(y)),
            "mod_x_type": "mRNA expression", "mod_y_type": "DNA methylation"}
    return (rna.values.astype(np.float64), meth.values.astype(np.float64), y,
            list(rna.columns), list(meth.columns), info)


def load_mfeat_raw():
    """UCI Multiple Features (mfeat) — classic REAL multi-view benchmark.

    2000 handwritten digits (10 balanced classes), where each *view* is a
    different feature-extraction modality computed by the original authors.
    We use the two most complementary strong views as the two modalities:
      mod_x = profile correlations   ("fac", 216 features)
      mod_y = Fourier coefficients   ("fou",  76 features)
    Files ship inside the `mvlearn` package (pip install mvlearn), so no
    network access is needed. Values are raw; per-fold standardisation is
    applied by FoldPreprocessor as for every other dataset.
    Reference: Breukelen et al. (1998), Kybernetika 34:381; UCI repository
    dataset 72 ("Multiple Features"); van der Maaten benchmark lineage.
    """
    from mvlearn.datasets import load_UCImultifeature
    Xs, y = load_UCImultifeature()
    # mvlearn view order: fou(76), fac(216), kar(64), pix(240), zer(47), mor(6)
    fou, fac = Xs[0], Xs[1]
    y = y.astype(np.int64)
    info = {"name": "MFEAT", "n": len(y), "classes": len(np.unique(y)),
            "mod_x_type": "Profile correlations (fac)",
            "mod_y_type": "Fourier coefficients (fou)"}
    return (fac.astype(np.float64), fou.astype(np.float64), y,
            [f"fac_{i}" for i in range(fac.shape[1])],
            [f"fou_{i}" for i in range(fou.shape[1])], info)


## 4. BNF Grammar (C2, C5)

Compact search space (~1.8×10⁷ architectures) where **every gene is functional**: per-modality encoder depth/width/dropout/norm/activation, embedding dim, fusion operator, head shape.

In [ ]:
class EvoFuseGrammar:
    """Genotype (list of int codons) -> phenotype (architecture description).

    Search space ≈ 1.8e7 distinct architectures: per-modality encoder
    (depth, width, dropout, norm, activation), shared embedding dim, fusion
    operator, and classification head (depth, width multiplier, dropout).
    """

    RULES = {
        "enc_layers":  [0, 1, 2],
        "enc_width":   [16, 32, 64, 128],
        "dropout":     [0.0, 0.1, 0.25, 0.4],
        "norm":        ["none", "batch", "layer"],
        "act":         ["relu", "gelu"],
        "emb_dim":     [16, 32, 64],
        "fusion":      ["concat", "sum", "gmu", "crossgate"],
        "head_layers": [0, 1, 2],
        "head_mult":   [1, 2],
        "head_drop":   [0.0, 0.15, 0.3],
    }

    def decode(self, codons):
        it = iter(range(len(codons) * 4))  # wrap-around reads

        def nxt(rule):
            i = next(it)
            return self.RULES[rule][codons[i % len(codons)] % len(self.RULES[rule])]

        def enc():
            return {"layers": nxt("enc_layers"), "width": nxt("enc_width"),
                    "dropout": nxt("dropout"), "norm": nxt("norm"), "act": nxt("act")}

        return {
            "enc_x": enc(), "enc_y": enc(),
            "emb_dim": nxt("emb_dim"), "fusion": nxt("fusion"),
            "head": {"layers": nxt("head_layers"), "mult": nxt("head_mult"),
                     "dropout": nxt("head_drop")},
        }

    @staticmethod
    def canonical(pheno) -> str:
        """Stable key for phenotype-level fitness caching (C8)."""
        return json.dumps(pheno, sort_keys=True)

    @staticmethod
    def to_string(p):
        e = lambda d: f"[{d['layers']}x{d['width']} {d['act']}/{d['norm']} do={d['dropout']}]"
        return (f"encX{e(p['enc_x'])} encY{e(p['enc_y'])} emb={p['emb_dim']} "
                f"fuse={p['fusion']} head=[{p['head']['layers']}x*{p['head']['mult']} "
                f"do={p['head']['dropout']}]")

## 5. Fusion Network (C2)

End-to-end trainable: `Encoder(x)`, `Encoder(y)` → fusion (`concat`/`sum`/`gmu`/`crossgate`) → head. 3k–40k parameters.

In [ ]:
def _norm(kind, dim):
    return {"none": nn.Identity(), "batch": nn.BatchNorm1d(dim),
            "layer": nn.LayerNorm(dim)}[kind]


def _act(kind):
    return {"relu": nn.ReLU(), "gelu": nn.GELU()}[kind]


class Encoder(nn.Module):
    def __init__(self, in_dim, spec, emb_dim):
        super().__init__()
        layers, d = [], in_dim
        for _ in range(spec["layers"]):
            layers += [nn.Linear(d, spec["width"]), _norm(spec["norm"], spec["width"]),
                       _act(spec["act"]), nn.Dropout(spec["dropout"])]
            d = spec["width"]
        layers += [nn.Linear(d, emb_dim), _act(spec["act"])]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ConcatFusion(nn.Module):
    def __init__(self, d):
        super().__init__(); self.proj = nn.Linear(2 * d, d)
    def forward(self, hx, hy):
        return F.relu(self.proj(torch.cat([hx, hy], -1)))


class SumFusion(nn.Module):
    def __init__(self, d):
        super().__init__(); self.w = nn.Parameter(torch.zeros(2))
    def forward(self, hx, hy):
        w = torch.softmax(self.w, 0)
        return w[0] * hx + w[1] * hy


class GMUFusion(nn.Module):
    """Gated Multimodal Unit (Arevalo et al.)."""
    def __init__(self, d):
        super().__init__()
        self.tx, self.ty = nn.Linear(d, d), nn.Linear(d, d)
        self.gate = nn.Linear(2 * d, d)
    def forward(self, hx, hy):
        g = torch.sigmoid(self.gate(torch.cat([hx, hy], -1)))
        return g * torch.tanh(self.tx(hx)) + (1 - g) * torch.tanh(self.ty(hy))


class CrossGateFusion(nn.Module):
    """Each modality is modulated by a gate computed from the other."""
    def __init__(self, d):
        super().__init__(); self.gx, self.gy = nn.Linear(d, d), nn.Linear(d, d)
    def forward(self, hx, hy):
        return hx * torch.sigmoid(self.gy(hy)) + hy * torch.sigmoid(self.gx(hx))


FUSIONS = {"concat": ConcatFusion, "sum": SumFusion,
           "gmu": GMUFusion, "crossgate": CrossGateFusion}


class EvoFuseNet(nn.Module):
    """Encoder-per-modality -> fusion -> head. Typical size 3k-40k params —
    matched to n≈10^2-10^3 samples (v3's frozen-backbone fusion nets exceeded
    100k trainable params on 30 input features)."""

    def __init__(self, dim_x, dim_y, pheno, n_classes):
        super().__init__()
        d = pheno["emb_dim"]
        self.enc_x = Encoder(dim_x, pheno["enc_x"], d)
        self.enc_y = Encoder(dim_y, pheno["enc_y"], d)
        self.fusion = FUSIONS[pheno["fusion"]](d)
        h, hd = pheno["head"], d * pheno["head"]["mult"]
        layers, dd = [], d
        for _ in range(h["layers"]):
            layers += [nn.Linear(dd, hd), nn.ReLU(), nn.Dropout(h["dropout"])]
            dd = hd
        layers += [nn.Linear(dd, n_classes)]
        self.head = nn.Sequential(*layers)

    def forward(self, mx, my):
        return self.head(self.fusion(self.enc_x(mx), self.enc_y(my)))

## 6. Fitness Evaluation (C3, C6, C7, C8)

Mean balanced accuracy over 3 fixed stratified inner splits; identical splits for every candidate (paired comparison); deterministic per-(phenotype, split) seeds; phenotype-level cache; failures logged, never silent.

In [ ]:
def train_fusion_model(pheno, Xtr, Ytr, ytr, Xva, Yva, yva, n_classes, cfg, seed):
    """Train one EvoFuseNet end-to-end with early stopping on balanced accuracy.
    Returns (best_val_balacc, model_with_best_weights)."""
    set_seed(seed)
    dev = torch.device(cfg["device"])
    model = EvoFuseNet(Xtr.shape[1], Ytr.shape[1], pheno, n_classes).to(dev)
    counts = np.bincount(ytr, minlength=n_classes).astype(np.float64)
    w = counts.sum() / np.maximum(counts, 1) / n_classes
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=dev),
                               label_smoothing=cfg.get("label_smoothing", 0.0))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                            weight_decay=cfg["weight_decay"])
    tXtr = torch.tensor(Xtr, device=dev); tYtr = torch.tensor(Ytr, device=dev)
    tytr = torch.tensor(ytr, device=dev)
    tXva = torch.tensor(Xva, device=dev); tYva = torch.tensor(Yva, device=dev)
    n, bs = len(ytr), cfg["batch_size"]
    max_epochs = cfg.get("_max_epochs", cfg["proxy_max_epochs"])
    patience = cfg.get("_patience", cfg["proxy_patience"])
    best_ba, best_state, wait = -1.0, None, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(max_epochs):
        model.train()
        for idx in torch.randperm(n, generator=g).split(bs):
            opt.zero_grad()
            loss = crit(model(tXtr[idx], tYtr[idx]), tytr[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(tXva, tYva).argmax(1).cpu().numpy()
        ba = balanced_accuracy_score(yva, pv)
        if ba > best_ba + 1e-6:
            best_ba, wait = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return best_ba, model


class FitnessEvaluator:
    """Fitness(pheno) = mean balanced accuracy over `inner_splits` fixed
    stratified train/val splits of the outer-training fold.

    The splits are IDENTICAL for every candidate (paired comparison), and each
    (phenotype, split) training run is deterministically seeded, so fitness is
    a pure function of the phenotype — evaluated once, then cached.
    """

    def __init__(self, Xtr, Ytr, ytr, n_classes, cfg, fold_seed):
        self.X, self.Y, self.y = Xtr, Ytr, ytr
        self.nc, self.cfg, self.fold_seed = n_classes, cfg, fold_seed
        sss = StratifiedShuffleSplit(n_splits=cfg["inner_splits"],
                                     test_size=cfg["inner_val_frac"],
                                     random_state=fold_seed)
        self.splits = list(sss.split(Xtr, ytr))
        self.cache, self.n_evals, self.n_failures = {}, 0, 0
        self.failure_log = []

    def evaluate(self, pheno):
        key = EvoFuseGrammar.canonical(pheno)
        if key in self.cache:
            return self.cache[key]
        self.n_evals += 1
        scores = []
        try:
            for si, (tr, va) in enumerate(self.splits):
                seed = (self.fold_seed * 1000 + si * 97 +
                        int(hashlib.md5(key.encode()).hexdigest()[:6], 16)) % (2**31)
                ba, model = train_fusion_model(
                    pheno, self.X[tr], self.Y[tr], self.y[tr],
                    self.X[va], self.Y[va], self.y[va], self.nc, self.cfg, seed)
                scores.append(ba)
                del model
            fit = float(np.mean(scores))
        except Exception as e:                       # C7: loud, not silent
            self.n_failures += 1
            self.failure_log.append({"pheno": pheno, "error": repr(e)})
            print(f"    !! evaluation failure #{self.n_failures}: {repr(e)[:120]}")
            fit = 0.0
        self.cache[key] = fit
        return fit

## 7. Grammatical Evolution Engine (C4, C8)

Elitism + tournament + two-point crossover + per-codon mutation. Returns the **top-k distinct** architectures. `random_search` is the equal-budget control with the same selection protocol.

In [ ]:
class GESearch:
    """GE over codon genomes with elitism, tournament selection, two-point
    crossover and per-codon mutation. Returns the top-k DISTINCT architectures
    (for ensembling) rather than a single best individual."""

    def __init__(self, cfg, evaluator, rng=None):
        self.cfg, self.ev = cfg, evaluator
        self.grammar = EvoFuseGrammar()
        self.rng = rng or random.Random(cfg["base_seed"])
        self.history = []

    def _rand_geno(self):
        return [self.rng.randrange(self.cfg["codon_size"])
                for _ in range(self.cfg["genotype_length"])]

    def _fitness(self, geno):
        return self.ev.evaluate(self.grammar.decode(geno))

    def _tournament(self, pop, fits):
        idx = max(self.rng.sample(range(len(pop)), self.cfg["tournament_size"]),
                  key=lambda i: fits[i])
        return pop[idx]

    def _crossover(self, g1, g2):
        if self.rng.random() > self.cfg["crossover_rate"]:
            return g1[:], g2[:]
        a, b = sorted(self.rng.sample(range(1, len(g1)), 2))
        return g1[:a] + g2[a:b] + g1[b:], g2[:a] + g1[a:b] + g2[b:]

    def _mutate(self, g):
        return [self.rng.randrange(self.cfg["codon_size"])
                if self.rng.random() < self.cfg["codon_mutation_rate"] else c
                for c in g]

    def run(self, verbose=True):
        pop = [self._rand_geno() for _ in range(self.cfg["population_size"])]
        fits = [self._fitness(g) for g in pop]
        best, wait = max(fits), 0
        for gen in range(self.cfg["max_generations"]):
            t0 = time.time()
            order = sorted(range(len(pop)), key=lambda i: fits[i], reverse=True)
            new_pop = [pop[i][:] for i in order[:self.cfg["elite_size"]]]
            while len(new_pop) < self.cfg["population_size"]:
                c1, c2 = self._crossover(self._tournament(pop, fits),
                                         self._tournament(pop, fits))
                new_pop += [self._mutate(c1), self._mutate(c2)]
            pop = new_pop[:self.cfg["population_size"]]
            fits = [self._fitness(g) for g in pop]
            gen_best = max(fits)
            self.history.append({"gen": gen + 1, "best": gen_best,
                                 "mean": float(np.mean(fits)),
                                 "unique_evals": self.ev.n_evals,
                                 "time": time.time() - t0})
            if verbose:
                h = self.history[-1]
                print(f"    gen {gen+1:2d}: best={h['best']:.4f} "
                      f"mean={h['mean']:.4f} evals={h['unique_evals']} "
                      f"({h['time']:.0f}s)")
            if gen_best > best + 1e-4:
                best, wait = gen_best, 0
            else:
                wait += 1
                if wait >= self.cfg["ge_patience"]:
                    if verbose:
                        print(f"    early stop at generation {gen+1}")
                    break
        return self.top_k(self.cfg["ensemble_k"])

    def top_k(self, k):
        """Top-k distinct phenotypes from the evaluation cache (C4)."""
        items = sorted(self.ev.cache.items(), key=lambda kv: kv[1], reverse=True)
        return [(json.loads(key), fit) for key, fit in items[:k]]


def random_search(cfg, evaluator, budget, seed):
    """Control with the SAME unique-evaluation budget and selection protocol."""
    rng = random.Random(seed)
    grammar = EvoFuseGrammar()
    while evaluator.n_evals < budget:
        evaluator.evaluate(grammar.decode(
            [rng.randrange(cfg["codon_size"]) for _ in range(cfg["genotype_length"])]))
    items = sorted(evaluator.cache.items(), key=lambda kv: kv[1], reverse=True)
    return [(json.loads(k), f) for k, f in items[:cfg["ensemble_k"]]]

## 8. Final Predictor: Top-k Ensemble (C4)

In [ ]:
def fit_predict_ensemble(top_phenos, Xtr, Ytr, ytr, Xte, Yte, n_classes, cfg,
                         fold_seed):
    """Retrain each top architecture on the outer-training fold (with a small
    early-stopping split) under `ensemble_seeds` seeds; average the softmax."""
    sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg["final_val_frac"],
                                 random_state=fold_seed + 1)
    tr, va = next(sss.split(Xtr, ytr))
    dev = torch.device(cfg["device"])
    cfg_final = dict(cfg, _max_epochs=cfg["final_max_epochs"],
                     _patience=cfg["final_patience"])
    probs, n_params = [], []
    tXte = torch.tensor(Xte, device=dev); tYte = torch.tensor(Yte, device=dev)
    for ai, (pheno, fit) in enumerate(top_phenos):
        for s in range(cfg["ensemble_seeds"]):
            seed = fold_seed * 10000 + ai * 100 + s
            _, model = train_fusion_model(
                pheno, Xtr[tr], Ytr[tr], ytr[tr], Xtr[va], Ytr[va], ytr[va],
                n_classes, cfg_final, seed)
            with torch.no_grad():
                probs.append(F.softmax(model(tXte, tYte), 1).cpu().numpy())
            n_params.append(sum(p.numel() for p in model.parameters()))
            del model
    return np.mean(probs, axis=0), int(np.mean(n_params))


def classification_metrics(y_true, probs):
    preds = probs.argmax(1)
    out = {"acc": accuracy_score(y_true, preds),
           "bal_acc": balanced_accuracy_score(y_true, preds),
           "f1": f1_score(y_true, preds, average="macro")}
    try:
        if probs.shape[1] == 2:
            out["auc"] = roc_auc_score(y_true, probs[:, 1])
        else:
            out["auc"] = roc_auc_score(y_true, probs, multi_class="ovr",
                                       average="macro")
    except Exception:
        out["auc"] = float("nan")
    return out

## 9. Baselines

13 baselines (classical ML, unimodal MLPs, fusion NNs). All share the same fold preprocessing; NN baselines get the same early stopping and class weighting as EvoFuse — no strawmen.

In [ ]:
def get_sklearn_baselines(seed=42):
    b = {"Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                                   random_state=seed),
         "SVM (RBF)": SVC(probability=True, class_weight="balanced", random_state=seed),
         "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                 random_state=seed, n_jobs=-1),
         "Gradient Boosting": GradientBoostingClassifier(n_estimators=200,
                                                         random_state=seed)}
    if HAS_XGB:
        b["XGBoost"] = xgb.XGBClassifier(n_estimators=300, eval_metric="logloss",
                                         random_state=seed, n_jobs=2, verbosity=0)
    if HAS_LGB:
        b["LightGBM"] = lgb.LGBMClassifier(n_estimators=300, random_state=seed,
                                           verbose=-1, n_jobs=2)
    return b


class _FixedFusionNet(nn.Module):
    """Shared wrapper so NN baselines reuse the same training loop as EvoFuse."""
    def __init__(self, kind, dim_x, dim_y, nc, hidden=64):
        super().__init__()
        self.kind = kind
        if kind == "early":
            self.net = nn.Sequential(nn.Linear(dim_x + dim_y, 2 * hidden), nn.ReLU(),
                                     nn.Dropout(0.3), nn.Linear(2 * hidden, hidden),
                                     nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, nc))
        elif kind in ("x_only", "y_only"):
            d = dim_x if kind == "x_only" else dim_y
            self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(0.3),
                                     nn.Linear(hidden, hidden // 2), nn.ReLU(),
                                     nn.Dropout(0.2), nn.Linear(hidden // 2, nc))
        elif kind == "late":
            self.ex = nn.Sequential(nn.Linear(dim_x, hidden), nn.ReLU(), nn.Dropout(0.2))
            self.ey = nn.Sequential(nn.Linear(dim_y, hidden), nn.ReLU(), nn.Dropout(0.2))
            self.head = nn.Linear(hidden, nc)
        elif kind == "intermediate":
            self.ex = nn.Sequential(nn.Linear(dim_x, hidden), nn.ReLU(), nn.Dropout(0.2))
            self.ey = nn.Sequential(nn.Linear(dim_y, hidden), nn.ReLU(), nn.Dropout(0.2))
            self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                      nn.Dropout(0.2), nn.Linear(hidden, nc))
        elif kind == "gmu":
            self.ex = nn.Sequential(nn.Linear(dim_x, hidden), nn.Tanh())
            self.ey = nn.Sequential(nn.Linear(dim_y, hidden), nn.Tanh())
            self.gate = nn.Sequential(nn.Linear(dim_x + dim_y, hidden), nn.Sigmoid())
            self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(),
                                      nn.Dropout(0.2), nn.Linear(hidden // 2, nc))

    def forward(self, mx, my):
        k = self.kind
        if k == "early":
            return self.net(torch.cat([mx, my], -1))
        if k == "x_only":
            return self.net(mx)
        if k == "y_only":
            return self.net(my)
        if k == "late":
            return self.head((self.ex(mx) + self.ey(my)) / 2)
        if k == "intermediate":
            return self.head(torch.cat([self.ex(mx), self.ey(my)], -1))
        g = self.gate(torch.cat([mx, my], -1))
        return self.head(g * self.ex(mx) + (1 - g) * self.ey(my))


def train_nn_baseline(kind, Xtr, Ytr, ytr, Xte, Yte, n_classes, cfg, seed):
    """NN baselines get the same early stopping + class weighting as EvoFuse."""
    set_seed(seed)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg["final_val_frac"],
                                 random_state=seed)
    tr, va = next(sss.split(Xtr, ytr))
    dev = torch.device(cfg["device"])
    model = _FixedFusionNet(kind, Xtr.shape[1], Ytr.shape[1], n_classes).to(dev)
    counts = np.bincount(ytr[tr], minlength=n_classes).astype(np.float64)
    w = counts.sum() / np.maximum(counts, 1) / n_classes
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=dev))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                            weight_decay=cfg["weight_decay"])
    tX, tY = torch.tensor(Xtr[tr], device=dev), torch.tensor(Ytr[tr], device=dev)
    ty = torch.tensor(ytr[tr], device=dev)
    vX, vY = torch.tensor(Xtr[va], device=dev), torch.tensor(Ytr[va], device=dev)
    n, bs = len(tr), cfg["batch_size"]
    best_ba, best_state, wait = -1, None, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(cfg["final_max_epochs"]):
        model.train()
        for idx in torch.randperm(n, generator=g).split(bs):
            opt.zero_grad()
            crit(model(tX[idx], tY[idx]), ty[idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            ba = balanced_accuracy_score(ytr[va], model(vX, vY).argmax(1).cpu().numpy())
        if ba > best_ba + 1e-6:
            best_ba, wait = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= cfg["final_patience"]:
                break
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(torch.tensor(Xte, device=dev),
                                torch.tensor(Yte, device=dev)), 1).cpu().numpy()
    return probs

## 10. Cross-Validation Protocol & Statistics

Repeated stratified k-fold; per-fold checkpointing; Wilcoxon signed-rank vs EvoFuse with Bonferroni correction.

In [ ]:
def run_single_fold(dset, train_idx, test_idx, fold_id, cfg, verbose=True):
    """One outer fold: leak-free preprocessing, EvoFuse search + ensemble,
    equal-budget random search, all baselines. Returns {method: metrics}."""
    mod_x, mod_y, y = dset["mod_x"], dset["mod_y"], dset["labels"]
    nc = dset["num_classes"]
    fold_seed = cfg["base_seed"] + fold_id

    pre = FoldPreprocessor().fit(mod_x[train_idx], mod_y[train_idx])   # C1
    Xtr, Ytr = pre.transform(mod_x[train_idx], mod_y[train_idx])
    Xte, Yte = pre.transform(mod_x[test_idx], mod_y[test_idx])
    ytr, yte = y[train_idx], y[test_idx]
    results = {}

    # ---- EvoFuse ----
    t0 = time.time()
    ev = FitnessEvaluator(Xtr, Ytr, ytr, nc, cfg, fold_seed)
    ge = GESearch(cfg, ev, rng=random.Random(fold_seed))
    top = ge.run(verbose=verbose)
    ge_budget = ev.n_evals
    probs, npar = fit_predict_ensemble(top, Xtr, Ytr, ytr, Xte, Yte, nc, cfg,
                                       fold_seed)
    results["EvoFuse"] = {**classification_metrics(yte, probs),
                          "num_params": npar, "search_evals": ge_budget,
                          "search_time_s": time.time() - t0,
                          "top_pheno": top[0][0], "top_fitness": top[0][1],
                          "history": ge.history, "n_eval_failures": ev.n_failures}
    if verbose:
        print(f"    EvoFuse: bal_acc={results['EvoFuse']['bal_acc']:.4f} "
              f"acc={results['EvoFuse']['acc']:.4f} "
              f"({ge_budget} evals, {time.time()-t0:.0f}s)")

    # ---- Random search control (identical budget + protocol, C8) ----
    ev_rs = FitnessEvaluator(Xtr, Ytr, ytr, nc, cfg, fold_seed)
    top_rs = random_search(cfg, ev_rs, ge_budget, fold_seed + 555)
    probs_rs, _ = fit_predict_ensemble(top_rs, Xtr, Ytr, ytr, Xte, Yte, nc, cfg,
                                       fold_seed)
    results["Random Search"] = classification_metrics(yte, probs_rs)

    # ---- Neural baselines ----
    for name, kind in [("MLP (Early Fusion)", "early"), ("MLP (Mod X Only)", "x_only"),
                       ("MLP (Mod Y Only)", "y_only"), ("Late Fusion MLP", "late"),
                       ("Intermediate Fusion", "intermediate"),
                       ("Gated Multimodal", "gmu")]:
        probs_b = train_nn_baseline(kind, Xtr, Ytr, ytr, Xte, Yte, nc, cfg,
                                    fold_seed + 17)
        results[name] = classification_metrics(yte, probs_b)

    # ---- Classical ML baselines ----
    Xc_tr, Xc_te = np.hstack([Xtr, Ytr]), np.hstack([Xte, Yte])
    for name, model in get_sklearn_baselines(fold_seed).items():
        model.fit(Xc_tr, ytr)
        pb = (model.predict_proba(Xc_te) if hasattr(model, "predict_proba")
              else np.eye(nc)[model.predict(Xc_te)])
        results[name] = classification_metrics(yte, pb)

    return results


def run_dataset_experiment(name, dset, cfg, out_dir=None, verbose=True):
    rskf = RepeatedStratifiedKFold(n_splits=cfg["n_outer_folds"],
                                   n_repeats=cfg["n_repetitions"],
                                   random_state=cfg["base_seed"])
    all_folds = []
    for fi, (tr, te) in enumerate(rskf.split(dset["mod_x"], dset["labels"])):
        if verbose:
            print(f"\n  -- {name} fold {fi+1}/{cfg['n_outer_folds']*cfg['n_repetitions']} --")
        set_seed(cfg["base_seed"] + fi)
        all_folds.append(run_single_fold(dset, tr, te, fi, cfg, verbose=verbose))
        if out_dir:
            ck = Path(out_dir) / "checkpoints"; ck.mkdir(parents=True, exist_ok=True)
            with open(ck / f"{name}_fold{fi}.json", "w") as f:
                json.dump(all_folds, f, default=str)
    return all_folds


def compute_statistics(all_folds, metric="bal_acc", ref="EvoFuse"):
    methods = sorted({m for fr in all_folds for m in fr})
    scores = {m: [fr[m][metric] if m in fr else np.nan for fr in all_folds]
              for m in methods}
    rows = []
    for m in methods:
        v = [s for s in scores[m] if not np.isnan(s)]
        p = np.nan
        if HAS_SCIPY and m != ref and ref in scores:
            paired = [(a, b) for a, b in zip(scores[ref], scores[m])
                      if not (np.isnan(a) or np.isnan(b))]
            if len(paired) >= 5 and any(a != b for a, b in paired):
                try:
                    _, p = sps.wilcoxon(*zip(*paired))
                except Exception:
                    pass
        rows.append({"Method": m, "Mean": np.mean(v), "Std": np.std(v),
                     "p-value": p, "n": len(v)})
    df = pd.DataFrame(rows).sort_values("Mean", ascending=False).reset_index(drop=True)
    n_comp = max(len(df) - 1, 1)
    df["p-corrected"] = df["p-value"].apply(
        lambda p: min(p * n_comp, 1.0) if not np.isnan(p) else np.nan)
    return df, scores

## 11. Self-Test (C5)

Proves each grammar gene changes the network, every fusion op runs, and the cache key is stable. Run before any experiment.

In [ ]:
def _self_test():
    g = EvoFuseGrammar()
    rng = random.Random(0)
    base = [rng.randrange(256) for _ in range(32)]
    p0 = g.decode(base)
    net = EvoFuseNet(18, 12, p0, 2)
    out = net(torch.randn(4, 18), torch.randn(4, 12))
    assert out.shape == (4, 2)
    # every fusion op builds and runs
    for f in FUSIONS:
        p = copy.deepcopy(p0); p["fusion"] = f
        assert EvoFuseNet(18, 12, p, 2)(torch.randn(4, 18),
                                        torch.randn(4, 12)).shape == (4, 2)
    # phenotype canonical key is stable and distinct phenotypes differ
    p1 = copy.deepcopy(p0); p1["emb_dim"] = 16 if p0["emb_dim"] != 16 else 32
    assert EvoFuseGrammar.canonical(p0) != EvoFuseGrammar.canonical(p1)
    assert EvoFuseGrammar.canonical(p0) == EvoFuseGrammar.canonical(
        g.decode(base))
    # every gene position influences the phenotype for some codon change
    changed = 0
    for i in range(15):  # first 15 codons cover one full decode pass
        for delta in range(1, 5):
            mut = base[:]; mut[i] = (mut[i] + delta) % 256
            if EvoFuseGrammar.canonical(g.decode(mut)) != EvoFuseGrammar.canonical(p0):
                changed += 1
                break
    assert changed >= 13, f"only {changed}/15 leading codons are functional"
    print("EvoFuse self-test passed: all fusion ops run, "
          f"{changed}/15 leading codons verified functional, caching key stable.")


if __name__ == "__main__":
    _self_test()

In [ ]:
_self_test()

## 12. Load Datasets

In [ ]:
# Dataset registry: real loaders; TCGA needs your local Datasets/ directory.
DATASET_CONFIG = {
    "WBCD":       {"enabled": True},
    "KIPAN":      {"enabled": True},   # real multi-omics (mRNA + DNAm)
    "MFEAT":      {"enabled": True},   # real multi-view (profile-corr + Fourier)
    "TCGA-BRCA":  {"enabled": False},
    "TCGA-LUAD":  {"enabled": False},
    "TCGA-LUSC":  {"enabled": False},
}
ALLOW_SYNTHETIC_FALLBACK = False   # True only for pipeline testing — synthetic
                                   # results are never comparable to real ones.

ALL_DATASETS = {}
for name, dc in DATASET_CONFIG.items():
    if not dc["enabled"]:
        continue
    try:
        if name == "WBCD":
            data = load_wbcd_raw()
        elif name == "KIPAN":
            data = load_kipan_raw()
        elif name == "MFEAT":
            data = load_mfeat_raw()
        else:
            data = load_tcga_raw(name)
            if data is None:
                if ALLOW_SYNTHETIC_FALLBACK:
                    print(f"{name}: local data not found -> SYNTHETIC placeholder")
                    data = make_synthetic_multiomics(name)
                else:
                    print(f"{name}: local data not found -> SKIPPED "
                          f"(set ALLOW_SYNTHETIC_FALLBACK=True to smoke-test)")
                    continue
    except Exception as e:
        print(f"{name}: load failed ({e}) -> skipped")
        continue
    mx, my, y, fx, fy, info = data
    ALL_DATASETS[name] = {"mod_x": mx, "mod_y": my, "labels": y, "info": info,
                          "fn_x": fx, "fn_y": fy,
                          "num_classes": len(np.unique(y))}
    cc = np.bincount(y)
    print(f"{name}: n={len(y)} dx={mx.shape[1]} dy={my.shape[1]} "
          f"classes={len(cc)} imbalance={cc.max()/max(cc.min(),1):.1f}x")
print(f"\nLoaded: {list(ALL_DATASETS)}")

## 13. Run Full Experiment

In [ ]:
# ===============================================================
# EXPERIMENT MODE
#   "quick"    -> pipeline test        (3 folds, pop 12 x 8 gen)
#   "standard" -> solid results        (5 folds, pop 20 x 12 gen)
#   "full"     -> publication quality  (5 folds x 3 reps, pop 24 x 15 gen)
# ===============================================================
EXPERIMENT_MODE = "standard"

if EXPERIMENT_MODE == "quick":
    CFG.update({"n_outer_folds": 3, "n_repetitions": 1,
                "population_size": 12, "max_generations": 8, "ge_patience": 3})
elif EXPERIMENT_MODE == "standard":
    CFG.update({"n_outer_folds": 5, "n_repetitions": 1,
                "population_size": 20, "max_generations": 12, "ge_patience": 4})
elif EXPERIMENT_MODE == "full":
    CFG.update({"n_outer_folds": 5, "n_repetitions": 3,
                "population_size": 24, "max_generations": 15, "ge_patience": 5})

EXPERIMENT_ID = f"EvoFuse_v4_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path(f"results/{EXPERIMENT_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Mode={EXPERIMENT_MODE}  folds={CFG['n_outer_folds']}x{CFG['n_repetitions']}  "
      f"GE=pop{CFG['population_size']}x{CFG['max_generations']}gen  -> {OUTPUT_DIR}")

EXPERIMENT_RESULTS = {}
t_start = time.time()
for ds_name, ds in ALL_DATASETS.items():
    print(f"\n{'='*70}\nDATASET: {ds_name} (n={ds['info']['n']})\n{'='*70}")
    try:
        EXPERIMENT_RESULTS[ds_name] = run_dataset_experiment(
            ds_name, ds, CFG, out_dir=OUTPUT_DIR)
    except Exception as e:
        print(f"ERROR in {ds_name}: {e} — continuing")
total_time = time.time() - t_start
print(f"\nEXPERIMENT COMPLETE — {total_time/60:.1f} min")

## 14. Results Summary with Statistical Tests

In [ ]:
all_stat_dfs = {}
for ds_name, folds in EXPERIMENT_RESULTS.items():
    print(f"\n{'─'*78}\nDataset: {ds_name}   (Wilcoxon vs EvoFuse, Bonferroni-corrected)\n{'─'*78}")
    for metric in ["bal_acc", "acc", "f1", "auc"]:
        df, _ = compute_statistics(folds, metric)
        all_stat_dfs[(ds_name, metric)] = df
    df = all_stat_dfs[(ds_name, "bal_acc")]
    dfa = all_stat_dfs[(ds_name, "acc")]; dff = all_stat_dfs[(ds_name, "f1")]
    print(f"  {'Method':<22} {'BalAcc':>16} {'Acc':>8} {'F1':>8} {'p(corr)':>9}")
    for _, r in df.iterrows():
        m = r['Method']
        a = dfa.loc[dfa.Method == m, 'Mean'].values[0]
        f1v = dff.loc[dff.Method == m, 'Mean'].values[0]
        p = r['p-corrected']
        ps = '—' if np.isnan(p) else f"{p:.3f}"
        print(f"  {m:<22} {r['Mean']:.4f}±{r['Std']:.4f} {a:>8.4f} {f1v:>8.4f} {ps:>9}")

## 15. Figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.0)

# Fig 1: balanced-accuracy heatmap (methods x datasets)
if EXPERIMENT_RESULTS:
    methods = sorted({m for f in EXPERIMENT_RESULTS.values() for fr in f for m in fr})
    M = pd.DataFrame(index=methods, columns=list(EXPERIMENT_RESULTS), dtype=float)
    for ds, folds in EXPERIMENT_RESULTS.items():
        df, _ = compute_statistics(folds, "bal_acc")
        for _, r in df.iterrows():
            M.loc[r["Method"], ds] = r["Mean"]
    M = M.loc[M.mean(axis=1).sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(2 + 1.6*len(M.columns), 0.42*len(M)+1))
    sns.heatmap(M.astype(float), annot=True, fmt=".3f", cmap="viridis", ax=ax,
                cbar_kws={"label": "balanced accuracy"})
    ax.set_title("Balanced accuracy by method and dataset")
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/"fig1_heatmap.png", dpi=200); plt.show()

In [ ]:
# Fig 2: EvoFuse search convergence per dataset
for ds, folds in EXPERIMENT_RESULTS.items():
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for fi, fr in enumerate(folds):
        h = fr.get("EvoFuse", {}).get("history", [])
        if h:
            ax.plot([g["gen"] for g in h], [g["best"] for g in h],
                    alpha=0.7, label=f"fold {fi+1}")
    ax.set_xlabel("generation"); ax.set_ylabel("best fitness (bal. acc)")
    ax.set_title(f"EvoFuse convergence — {ds}"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/f"fig2_convergence_{ds}.png", dpi=200)
    plt.show()

In [ ]:
# Fig 3: evolved architecture component distribution
from collections import Counter
for ds, folds in EXPERIMENT_RESULTS.items():
    fus = Counter(); emb = Counter()
    for fr in folds:
        p = fr.get("EvoFuse", {}).get("top_pheno")
        if isinstance(p, str): p = json.loads(p.replace("'", '"'))
        if p: fus[p["fusion"]] += 1; emb[p["emb_dim"]] += 1
    if fus:
        fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
        axes[0].bar(list(fus), list(fus.values())); axes[0].set_title(f"{ds}: fusion op")
        axes[1].bar([str(k) for k in emb], list(emb.values())); axes[1].set_title("emb dim")
        plt.tight_layout(); plt.savefig(OUTPUT_DIR/f"fig3_arch_{ds}.png", dpi=200); plt.show()

## 16. Save All Results

In [ ]:
summary = {"experiment": "EvoFuse v4", "timestamp": datetime.now().isoformat(),
           "mode": EXPERIMENT_MODE,
           "config": {k: str(v) if not isinstance(v, (int, float, bool, str)) else v
                      for k, v in CFG.items()},
           "environment": {"torch": torch.__version__, "numpy": np.__version__,
                            "device": DEVICE},
           "datasets": list(ALL_DATASETS), "total_time_minutes": total_time/60}
for ds_name, folds in EXPERIMENT_RESULTS.items():
    df, _ = compute_statistics(folds, "bal_acc")
    summary[ds_name] = df.to_dict(orient="records")
    for metric in ["acc", "bal_acc", "f1", "auc"]:
        d2, _ = compute_statistics(folds, metric)
        d2.to_csv(OUTPUT_DIR / f"{ds_name}_{metric}_results.csv", index=False)
with open(OUTPUT_DIR / "results_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Saved -> {OUTPUT_DIR}")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name)